In [ ]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
# 1. Подключение к БД
conn = sqlite3.connect("restaurant.db")

# 2. SQL-запрос с объединением 3 таблиц
query = """
SELECT
    o.id AS order_id,
    o.created_at,
    oi.product_id,
    oi.quantity,
    p.name AS product_name,
    p.price
FROM 'order' o
JOIN order_item oi ON o.id = oi.order_id
JOIN product p ON oi.product_id = p.id
"""

# 3. Извлекаем данные в DataFrame
df = pd.read_sql(query, conn)

# 4. Сохраняем в .csv
df.to_csv("combined_data.csv", index=False)

# 5. Чтение обратно из CSV (для проверки)
df_check = pd.read_csv("combined_data.csv")

# 6. Проверка количества строк
print(f"Rows in DataFrame: {len(df_check)}")  # Должно быть 74818

# 7. Закрытие соединения
conn.close()

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
# группировка по продукту и подсчёт общего количества
top_products = df.groupby("product_name")["quantity"].sum().sort_values(ascending=False).head(10)

# построение круговой диаграммы
plt.figure(figsize=(8, 8))
plt.pie(top_products, labels=top_products.index, autopct='%1.1f%%', startangle=140)
plt.title("Top 10 products by quantity sold")
plt.axis("equal")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
import matplotlib.pyplot as plt

# 1. Добавим столбец item_price (price * quantity)
df["item_price"] = df["price"] * df["quantity"]

# 2. Группировка по product_name и сумма item_price
top_revenue = df.groupby("product_name")["item_price"].sum().sort_values(ascending=False).head(10)

# 3. Построение круговой диаграммы
plt.figure(figsize=(8, 8))
plt.pie(top_revenue, labels=top_revenue.index, autopct='%1.1f%%', startangle=140)
plt.title("Top 10 products by revenue")
plt.axis("equal")
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
import matplotlib.pyplot as plt

# Убедимся, что order_datetime в формате datetime
df["order_datetime"] = pd.to_datetime(df["order_datetime"])

# Добавим колонку с часом заказа
df["order_hour"] = df["order_datetime"].dt.hour

# Группировка по часу и сумма прибыли
hourly_profit = df.groupby("order_hour")["item_price"].sum()

# Построение графика
plt.figure(figsize=(10, 6))
hourly_profit.plot(kind="bar", color="skyblue")
plt.title("Profit by Order Hour")
plt.xlabel("Order Hour")
plt.ylabel("Total Revenue")
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
import matplotlib.pyplot as plt

# Убедимся, что дата в правильном формате
df["order_datetime"] = pd.to_datetime(df["order_datetime"])

# Извлекаем день недели
df["order_day"] = df["order_datetime"].dt.day_name()

# Группируем и суммируем
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
daily_income = df.groupby("order_day")["item_price"].sum().reindex(day_order)

# Визуализация
plt.figure(figsize=(10, 6))
daily_income.plot(kind="bar", color="coral")
plt.title("Total Income by Day of the Week")
plt.xlabel("Day of the Week")
plt.ylabel("Total Revenue")
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()
